<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 356 (delta 144), reused 93 (delta 93), pack-reused 178 (from 2)
Receiving objects: 100% (356/356), 1.93 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (199/199), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [6]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

X = df[feature_cols]
y = df['is_declining']

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

**Finding #4 — "The Freshness Multiplier."** The paper reports a 283:1 growth-to-decline
ratio for the 361+ day freshness bucket — but is transparent that this comes from just 1
declining page against 283 growing ones. My methodology question: with n=1 on the
declining side, is a ratio even the right statistic to report here, or would a confidence
interval (or simply "insufficient sample, no claim") communicate the uncertainty more
honestly than a headline number? The paper does flag this itself ("too small and too
unstable to treat as a headline multiplier"), which is the right instinct — my question is
whether the number should be shown at all if it's pre-disclaimed as unreliable, since a
reader skimming for numbers may still anchor on 283:1 despite the caveat.

**ML Appendix — "What Predicts Health."** The Random Forest predicts `health_score` using
features including `avg_position`, `impressions`, and `ctr` — but `health_score` is
explicitly defined earlier in the paper as a weighted composite of exactly those inputs
(impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts). My methodology
question: does an 80/20 holdout split protect this claim, given the target is partly
*definitionally* constructed from several of the features? A holdout split guards against
overfitting to noise, but it doesn't guard against a model "discovering" a relationship
that was true by construction rather than learned from data. The paper does flag this
directly ("importance is descriptive rather than causal"), which is exactly the right
caveat — I'm not disputing the finding, just naming precisely why a strong holdout score
here wouldn't be meaningful evidence of a *learned* pattern.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

**Before:** a naive random 80/20 split — the same style used in the paper's own ML
appendix (Random Forest, Logistic Regression, Decision Tree all use plain 80/20 splits
per the Methodology section) — lets the same client appear in both train and test.
**After:** a `GroupShuffleSplit` by `client_id`, so no client's pages leak across the
split. Same features, same model, same metric — only the split changes.

In [8]:
# BEFORE: naive random split (no group awareness)
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model_naive = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_naive.fit(X_train_naive, y_train_naive)
naive_preds = model_naive.predict(X_test_naive)

# AFTER: honest grouped split by client
groups = df['client_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_grouped.fit(X_train_g, y_train_g)
grouped_preds = model_grouped.predict(X_test_g)

before_after = pd.DataFrame({
    'split': ['naive_random (before)', 'grouped_by_client (after)'],
    'precision': [
        precision_score(y_test_naive, naive_preds),
        precision_score(y_test_g, grouped_preds),
    ],
    'recall': [
        recall_score(y_test_naive, naive_preds),
        recall_score(y_test_g, grouped_preds),
    ],
    'f1': [
        f1_score(y_test_naive, naive_preds),
        f1_score(y_test_g, grouped_preds),
    ],
})
print(before_after)


                       split  precision    recall        f1
0      naive_random (before)   0.900785  0.931365  0.915820
1  grouped_by_client (after)   0.805353  0.900680  0.850353


**Result:** precision drops from 0.901 (naive) to 0.805 (grouped) — a real ~10-point
gap. This confirms the naive split was letting the model partly learn client-specific
patterns rather than a signal that generalizes to a client it hasn't seen. 0.805 is the
number I'd actually report and stand behind; 0.901 was inflated by leakage.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

Same hunt as Week 3, applied to this notebook's final 28-feature set: check every
feature's correlation with `trend_pct` (the raw signal the label is derived from), and
confirm none of `trend_pct`/`trend_direction`/`is_declining` themselves are in the
feature list.

In [9]:
leak_corr = df[feature_cols].corrwith(df['trend_pct']).abs().sort_values(ascending=False)
print("correlation with trend_pct (the label's source signal):")
print(leak_corr.head(5))

label_cols = ['trend_pct', 'trend_direction', 'is_declining']
overlap = set(feature_cols) & set(label_cols)
print(f"\nlabel-derived columns present in feature set: {overlap}")  # should be empty set

LEAK_THRESHOLD = 0.9
still_leaky = leak_corr[leak_corr > LEAK_THRESHOLD]
print(f"\nfeatures exceeding leak threshold ({LEAK_THRESHOLD}): {still_leaky.to_dict()}")

correlation with trend_pct (the label's source signal):
impressions_last_30d     0.096737
avg_position             0.047248
days_with_impressions    0.031776
impressions_90d          0.024187
competition              0.014426
dtype: float64

label-derived columns present in feature set: set()

features exceeding leak threshold (0.9): {}


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

**Original (overreaching) claim:** "The model predicts which pages will decline in
search performance."

**Rewritten (safe):** "Under a client-grouped holdout, the model achieves 0.805
precision on flagging pages with a downward 30-day-vs-prior-30-day impression trend —
a directional, decision-support signal for prioritizing manual review, not a causal
prediction of future outcomes or a claim about Google's ranking algorithm."

**Why the rewrite matters:** "predicts which pages will decline" implies a forward-looking,
causal guarantee. What the model actually does is score an *already-observed* trend
pattern against a client-held-out validation set — a real, useful, but bounded claim.
The precision number itself only means something because it's paired with the specific
split design (grouped, not naive) and the specific label definition (a 30-day trend
threshold), and it should never be quoted without those two qualifiers attached.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.